In [1]:
import pandas as pd
import os 

In [15]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import SquareTable

# Define the base directory
base_dir = "../reasoning/power-seeking/gemma" 

# Define your columns
SCORE_COLUMN = "Score" 
ID_COLUMN = "Question" # Used alongside 'Multiplier' as the composite key

def load_agg_data(base_path):
    data_mapping = {}
    if not os.path.exists(base_path):
        print(f"Error: Directory '{base_path}' not found.")
        return data_mapping

    for subdir_name in os.listdir(base_path):
        subdir_path = os.path.join(base_path, subdir_name)
        if os.path.isdir(subdir_path):
            behavior_file = os.path.join(subdir_path, f"{subdir_name}_behavior_agg.csv")
            utility_file = os.path.join(subdir_path, f"{subdir_name}_utility_agg.csv")
            
            subdir_data = {}
            if os.path.exists(behavior_file):
                subdir_data['behavior'] = pd.read_csv(behavior_file)
            if os.path.exists(utility_file):
                subdir_data['utility'] = pd.read_csv(utility_file)
            
            if subdir_data:
                data_mapping[subdir_name] = subdir_data

    return data_mapping

if __name__ == "__main__":
    gemma_data = load_agg_data(base_dir)
    
    # --- PASS 1: Collect global bad composite keys: (Question, Multiplier) ---
    global_bad_behavior_keys = set()
    global_bad_utility_keys = set()
    
    for subdir_name, subdir_data in gemma_data.items():
        if 'behavior' in subdir_data:
            b_df = subdir_data['behavior']
            if SCORE_COLUMN in b_df.columns and ID_COLUMN in b_df.columns and 'Multiplier' in b_df.columns:
                # Find rows where score is 0
                bad_mask = b_df[SCORE_COLUMN] == 0
                # Zip the question and multiplier together into tuples and add to our set
                bad_tuples = set(zip(b_df.loc[bad_mask, ID_COLUMN], b_df.loc[bad_mask, 'Multiplier']))
                global_bad_behavior_keys.update(bad_tuples)
                
        if 'utility' in subdir_data:
            u_df = subdir_data['utility']
            if SCORE_COLUMN in u_df.columns and ID_COLUMN in u_df.columns and 'Multiplier' in u_df.columns:
                # Find rows where score is 0
                bad_mask = u_df[SCORE_COLUMN] == 0
                # Zip the question and multiplier together into tuples and add to our set
                bad_tuples = set(zip(u_df.loc[bad_mask, ID_COLUMN], u_df.loc[bad_mask, 'Multiplier']))
                global_bad_utility_keys.update(bad_tuples)

    print(f"Found {len(global_bad_behavior_keys)} unique (Question, Multiplier) pairs to drop globally in Behavior.")
    print(f"Found {len(global_bad_utility_keys)} unique (Question, Multiplier) pairs to drop globally in Utility.")

    # --- PASS 2: Filter by composite key and calculate means ---
    behavior_avg_scores = {}
    utility_avg_scores = {}
    
    for subdir_name, subdir_data in gemma_data.items():
        
        # Calculate averages for Behavior
        if 'behavior' in subdir_data:
            b_df = subdir_data['behavior']
            if SCORE_COLUMN in b_df.columns and ID_COLUMN in b_df.columns and 'Multiplier' in b_df.columns:
                # Create a multi-index from the current dataframe to check against our bad keys
                current_keys = pd.MultiIndex.from_arrays([b_df[ID_COLUMN], b_df['Multiplier']])
                mask = current_keys.isin(global_bad_behavior_keys)
                
                # Keep only rows that are NOT in the bad keys mask
                clean_b_df = b_df[~mask]
                b_avg = clean_b_df.groupby('Multiplier')[SCORE_COLUMN].mean().to_dict()
                behavior_avg_scores[subdir_name] = b_avg

        # Calculate averages for Utility
        if 'utility' in subdir_data:
            u_df = subdir_data['utility']
            if SCORE_COLUMN in u_df.columns and ID_COLUMN in u_df.columns and 'Multiplier' in u_df.columns:
                # Create a multi-index from the current dataframe to check against our bad keys
                current_keys = pd.MultiIndex.from_arrays([u_df[ID_COLUMN], u_df['Multiplier']])
                mask = current_keys.isin(global_bad_utility_keys)
                
                # Keep only rows that are NOT in the bad keys mask
                clean_u_df = u_df[~mask]
                u_avg = clean_u_df.groupby('Multiplier')[SCORE_COLUMN].mean().to_dict()
                utility_avg_scores[subdir_name] = u_avg

    FINAL_RESULT = {
        "Single Layer Ablation": ["gemma3-1b-3","gemma3-1b-13","gemma3-1b-23"],
        "Multi Layer Ablation": ["gemma3-1b-all","gemma3-1b-all-exp","gemma3-1b-all-exp-similarity-abs-exponent"],
    }
    FINAL_SUBDIR =  ["gemma3-1b-3","gemma3-1b-13","gemma3-1b-23","gemma3-1b-all","gemma3-1b-all-exp","gemma3-1b-all-exp-similarity-abs-exponent"]

    for key, card in FINAL_RESULT.items():
        print(f"\n{key}")
        print("=== BEHAVIOR AVERAGE SCORES ===")
        for subdir, scores in behavior_avg_scores.items():
            if subdir in card:
                print(f"{subdir}: {scores}")
            
        print("\n=== UTILITY AVERAGE SCORES ===")
        for subdir, scores in utility_avg_scores.items():
            if subdir in card:
                print(f"{subdir}: {scores}")

# Define your list of baseline directory names here
baseline_subdirs = ["gemma3-1b-13", "gemma3-1b-all"] # Update with your actual folder names

# Dictionary to hold nested results: {baseline: {mode: {comparison_subdir: results}}}
results_mcnemar = {}

for baseline_name in baseline_subdirs:
    results_mcnemar[baseline_name] = {'behavior': {}, 'utility': {}}
    
    for mode in ['behavior', 'utility']:
        # Use the appropriate global bad keys for the current mode
        bad_keys = global_bad_behavior_keys if mode == 'behavior' else global_bad_utility_keys
        
        # Ensure the baseline actually exists and has data for this mode
        if baseline_name not in gemma_data or mode not in gemma_data[baseline_name]:
            print(f"Warning: Baseline '{baseline_name}' missing '{mode}' data. Skipping.")
            continue
            
        # 1. Prepare the Baseline Data
        base_df = gemma_data[baseline_name][mode]
        base_keys = pd.MultiIndex.from_arrays([base_df[ID_COLUMN], base_df['Multiplier']])
        clean_base = base_df[~base_keys.isin(bad_keys)]
        
        # 2. Iterate through all other subdirectories to compare against the current baseline
        for subdir_name, subdir_data in gemma_data.items():
            if subdir_name == baseline_name:
                continue # Skip comparing the baseline to itself
                
            if mode in subdir_data:
                comp_df = subdir_data[mode]
                
                # Clean the comparison data
                comp_keys = pd.MultiIndex.from_arrays([comp_df[ID_COLUMN], comp_df['Multiplier']])
                clean_comp = comp_df[~comp_keys.isin(bad_keys)]
                
                # 3. Merge Baseline and Comparison on the composite key
                merged = pd.merge(
                    clean_base[[ID_COLUMN, 'Multiplier', SCORE_COLUMN]],
                    clean_comp[[ID_COLUMN, 'Multiplier', SCORE_COLUMN]],
                    on=[ID_COLUMN, 'Multiplier'],
                    suffixes=('_base', '_comp')
                )
                
                if not merged.empty:
                    # 4. Create Contingency Table
                    contingency_table = pd.crosstab(
                        merged[f'{SCORE_COLUMN}_base'], 
                        merged[f'{SCORE_COLUMN}_comp']
                    )
                    
                    # Force the table to be square (McNemar-Bowker requirement)
                    all_cats = sorted(set(contingency_table.index) | set(contingency_table.columns))
                    contingency_table = contingency_table.reindex(index=all_cats, columns=all_cats, fill_value=0)
                    
                    # 5. Perform the Test (requires at least a 2x2 table)
                    if contingency_table.shape[0] > 1: 
                        st = SquareTable(contingency_table.values)
                        test_result = st.symmetry()
                        
                        results_mcnemar[baseline_name][mode][subdir_name] = {
                            "statistic": test_result.statistic,
                            "p_value": test_result.pvalue,
                            "n": len(merged)
                        }
                    else:
                        results_mcnemar[baseline_name][mode][subdir_name] = {
                            "error": "Not enough variance (1x1 table)", 
                            "n": len(merged)
                        }

# --- Print Results ---
print("\n=== McNEMAR-BOWKER TEST RESULTS ===")
for baseline_name, modes in results_mcnemar.items():
    print(f"\n==================================================")
    print(f"BASELINE: {baseline_name}")
    print(f"==================================================")
    
    for mode in ['behavior', 'utility']:
        print(f"\n-- {mode.upper()} --")
        if not modes[mode]:
            print("  No comparisons made.")
        
        for subdir, res in modes[mode].items():
            if "error" in res:
                print(f"  vs {subdir}: {res['error']} (n={res['n']})")
            else:
                sig = "SIGNIFICANT shift" if res['p_value'] < 0.05 else "not significant"
                if subdir in FINAL_SUBDIR:
                    print(f"  vs {subdir}: Stat={res['statistic']:.4f}, p={res['p_value']:.4e} ({sig}, n={res['n']})")

Found 560 unique (Question, Multiplier) pairs to drop globally in Behavior.
Found 323 unique (Question, Multiplier) pairs to drop globally in Utility.

Single Layer Ablation
=== BEHAVIOR AVERAGE SCORES ===
gemma3-1b-13: {-2.0: 2.934065934065934, -1.5: 2.8979591836734695, -1.0: 3.0, -0.5: 3.116504854368932, 0.0: 3.231958762886598, 0.5: 3.303030303030303, 1.0: 3.5575757575757576, 1.5: 3.658385093167702, 2.0: 3.760233918128655}
gemma3-1b-3: {-2.0: 3.2967032967032965, -1.5: 3.3469387755102042, -1.0: 2.989130434782609, -0.5: 2.9223300970873787, 0.0: 3.231958762886598, 0.5: 3.2666666666666666, 1.0: 3.393939393939394, 1.5: 3.608695652173913, 2.0: 3.760233918128655}
gemma3-1b-23: {-2.0: 3.340659340659341, -1.5: 3.326530612244898, -1.0: 3.1847826086956523, -0.5: 3.1359223300970873, 0.0: 3.231958762886598, 0.5: 3.2303030303030305, 1.0: 3.2363636363636363, 1.5: 3.422360248447205, 2.0: 3.2046783625730995}

=== UTILITY AVERAGE SCORES ===
gemma3-1b-13: {-2.0: 3.7133333333333334, -1.5: 3.865497076023

In [6]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import SquareTable

# Define the base directory
base_dir = "../reasoning/power-seeking/llama" 

# Define your columns
SCORE_COLUMN = "Score" 
ID_COLUMN = "Question" # Used alongside 'Multiplier' as the composite key

def load_agg_data(base_path):
    data_mapping = {}
    if not os.path.exists(base_path):
        print(f"Error: Directory '{base_path}' not found.")
        return data_mapping

    for subdir_name in os.listdir(base_path):
        subdir_path = os.path.join(base_path, subdir_name)
        if os.path.isdir(subdir_path):
            behavior_file = os.path.join(subdir_path, f"{subdir_name}_behavior_agg.csv")
            utility_file = os.path.join(subdir_path, f"{subdir_name}_utility_agg.csv")
            
            subdir_data = {}
            if os.path.exists(behavior_file):
                subdir_data['behavior'] = pd.read_csv(behavior_file)
            if os.path.exists(utility_file):
                subdir_data['utility'] = pd.read_csv(utility_file)
            
            if subdir_data:
                data_mapping[subdir_name] = subdir_data

    return data_mapping

if __name__ == "__main__":
    gemma_data = load_agg_data(base_dir)
    
    # --- PASS 1: Collect global bad composite keys: (Question, Multiplier) ---
    global_bad_behavior_keys = set()
    global_bad_utility_keys = set()
    
    for subdir_name, subdir_data in gemma_data.items():
        if 'behavior' in subdir_data:
            b_df = subdir_data['behavior']
            if SCORE_COLUMN in b_df.columns and ID_COLUMN in b_df.columns and 'Multiplier' in b_df.columns:
                # Find rows where score is 0
                bad_mask = b_df[SCORE_COLUMN] == 0
                # Zip the question and multiplier together into tuples and add to our set
                bad_tuples = set(zip(b_df.loc[bad_mask, ID_COLUMN], b_df.loc[bad_mask, 'Multiplier']))
                global_bad_behavior_keys.update(bad_tuples)
                
        if 'utility' in subdir_data:
            u_df = subdir_data['utility']
            if SCORE_COLUMN in u_df.columns and ID_COLUMN in u_df.columns and 'Multiplier' in u_df.columns:
                # Find rows where score is 0
                bad_mask = u_df[SCORE_COLUMN] == 0
                # Zip the question and multiplier together into tuples and add to our set
                bad_tuples = set(zip(u_df.loc[bad_mask, ID_COLUMN], u_df.loc[bad_mask, 'Multiplier']))
                global_bad_utility_keys.update(bad_tuples)

    print(f"Found {len(global_bad_behavior_keys)} unique (Question, Multiplier) pairs to drop globally in Behavior.")
    print(f"Found {len(global_bad_utility_keys)} unique (Question, Multiplier) pairs to drop globally in Utility.")

    # --- PASS 2: Filter by composite key and calculate means ---
    behavior_avg_scores = {}
    utility_avg_scores = {}
    
    for subdir_name, subdir_data in gemma_data.items():
        
        # Calculate averages for Behavior
        if 'behavior' in subdir_data:
            b_df = subdir_data['behavior']
            if SCORE_COLUMN in b_df.columns and ID_COLUMN in b_df.columns and 'Multiplier' in b_df.columns:
                # Create a multi-index from the current dataframe to check against our bad keys
                current_keys = pd.MultiIndex.from_arrays([b_df[ID_COLUMN], b_df['Multiplier']])
                mask = current_keys.isin(global_bad_behavior_keys)
                
                # Keep only rows that are NOT in the bad keys mask
                clean_b_df = b_df[~mask]
                b_avg = clean_b_df.groupby('Multiplier')[SCORE_COLUMN].mean().to_dict()
                behavior_avg_scores[subdir_name] = b_avg

        # Calculate averages for Utility
        if 'utility' in subdir_data:
            u_df = subdir_data['utility']
            if SCORE_COLUMN in u_df.columns and ID_COLUMN in u_df.columns and 'Multiplier' in u_df.columns:
                # Create a multi-index from the current dataframe to check against our bad keys
                current_keys = pd.MultiIndex.from_arrays([u_df[ID_COLUMN], u_df['Multiplier']])
                mask = current_keys.isin(global_bad_utility_keys)
                
                # Keep only rows that are NOT in the bad keys mask
                clean_u_df = u_df[~mask]
                u_avg = clean_u_df.groupby('Multiplier')[SCORE_COLUMN].mean().to_dict()
                utility_avg_scores[subdir_name] = u_avg

    # --- Print the Results ---
    print("\n=== BEHAVIOR AVERAGE SCORES ===")
    for subdir, scores in behavior_avg_scores.items():
        print(f"{subdir}: {scores}")
        
    print("\n=== UTILITY AVERAGE SCORES ===")
    for subdir, scores in utility_avg_scores.items():
        print(f"{subdir}: {scores}")

# Define your list of baseline directory names here
baseline_subdirs = ["llama3-1-8b-16", "llama3-1-8b-all"] # Update with your actual folder names

# Dictionary to hold nested results: {baseline: {mode: {comparison_subdir: results}}}
results_mcnemar = {}

for baseline_name in baseline_subdirs:
    results_mcnemar[baseline_name] = {'behavior': {}, 'utility': {}}
    
    for mode in ['behavior', 'utility']:
        # Use the appropriate global bad keys for the current mode
        bad_keys = global_bad_behavior_keys if mode == 'behavior' else global_bad_utility_keys
        
        # Ensure the baseline actually exists and has data for this mode
        if baseline_name not in gemma_data or mode not in gemma_data[baseline_name]:
            print(f"Warning: Baseline '{baseline_name}' missing '{mode}' data. Skipping.")
            continue
            
        # 1. Prepare the Baseline Data
        base_df = gemma_data[baseline_name][mode]
        base_keys = pd.MultiIndex.from_arrays([base_df[ID_COLUMN], base_df['Multiplier']])
        clean_base = base_df[~base_keys.isin(bad_keys)]
        
        # 2. Iterate through all other subdirectories to compare against the current baseline
        for subdir_name, subdir_data in gemma_data.items():
            if subdir_name == baseline_name:
                continue # Skip comparing the baseline to itself
                
            if mode in subdir_data:
                comp_df = subdir_data[mode]
                
                # Clean the comparison data
                comp_keys = pd.MultiIndex.from_arrays([comp_df[ID_COLUMN], comp_df['Multiplier']])
                clean_comp = comp_df[~comp_keys.isin(bad_keys)]
                
                # 3. Merge Baseline and Comparison on the composite key
                merged = pd.merge(
                    clean_base[[ID_COLUMN, 'Multiplier', SCORE_COLUMN]],
                    clean_comp[[ID_COLUMN, 'Multiplier', SCORE_COLUMN]],
                    on=[ID_COLUMN, 'Multiplier'],
                    suffixes=('_base', '_comp')
                )
                
                if not merged.empty:
                    # 4. Create Contingency Table
                    contingency_table = pd.crosstab(
                        merged[f'{SCORE_COLUMN}_base'], 
                        merged[f'{SCORE_COLUMN}_comp']
                    )
                    
                    # Force the table to be square (McNemar-Bowker requirement)
                    all_cats = sorted(set(contingency_table.index) | set(contingency_table.columns))
                    contingency_table = contingency_table.reindex(index=all_cats, columns=all_cats, fill_value=0)
                    
                    # 5. Perform the Test (requires at least a 2x2 table)
                    if contingency_table.shape[0] > 1: 
                        st = SquareTable(contingency_table.values)
                        test_result = st.symmetry()
                        
                        results_mcnemar[baseline_name][mode][subdir_name] = {
                            "statistic": test_result.statistic,
                            "p_value": test_result.pvalue,
                            "n": len(merged)
                        }
                    else:
                        results_mcnemar[baseline_name][mode][subdir_name] = {
                            "error": "Not enough variance (1x1 table)", 
                            "n": len(merged)
                        }

# --- Print Results ---
print("\n=== McNEMAR-BOWKER TEST RESULTS ===")
for baseline_name, modes in results_mcnemar.items():
    print(f"\n==================================================")
    print(f"BASELINE: {baseline_name}")
    print(f"==================================================")
    
    for mode in ['behavior', 'utility']:
        print(f"\n-- {mode.upper()} --")
        if not modes[mode]:
            print("  No comparisons made.")
        
        for subdir, res in modes[mode].items():
            if "error" in res:
                print(f"  vs {subdir}: {res['error']} (n={res['n']})")
            else:
                sig = "SIGNIFICANT shift" if res['p_value'] < 0.05 else "not significant"
                print(f"  vs {subdir}: Stat={res['statistic']:.4f}, p={res['p_value']:.4e} ({sig}, n={res['n']})")

Found 373 unique (Question, Multiplier) pairs to drop globally in Behavior.
Found 194 unique (Question, Multiplier) pairs to drop globally in Utility.

=== BEHAVIOR AVERAGE SCORES ===
meta-llama_Llama-3.1-8B-Instruct: {0.0: 2.5380710659898478}
llama3-1-8b-all-exp-similarity-abs: {-2.0: 1.7, -1.5: 2.0, -1.0: 2.158273381294964, -0.5: 2.4545454545454546, 0.0: 2.5380710659898478, 0.5: 2.657608695652174, 1.0: 2.8633879781420766, 1.5: 3.2320441988950277, 2.0: 3.5}
llama3-1-8b-all-exp: {-2.0: 1.5083333333333333, -1.5: 1.856, -1.0: 2.0719424460431655, -0.5: 2.371212121212121, 0.0: 2.5380710659898478, 0.5: 2.766304347826087, 1.0: 2.9672131147540983, 1.5: 3.3812154696132595, 2.0: 3.897590361445783}
llama3-1-8b-all: {-2.0: 1.1333333333333333, -1.5: 1.128, -1.0: 1.2589928057553956, -0.5: 2.143939393939394, 0.0: 2.5380710659898478, 0.5: 2.5217391304347827, 1.0: 2.240437158469945, 1.5: 2.2209944751381214, 2.0: 2.4879518072289155}
llama3-1-8b-16: {-2.0: 2.3, -1.5: 2.304, -1.0: 2.4892086330935252, -0.

In [19]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import SquareTable

# Define the base directory
base_dir = "../reasoning/power-seeking/mistral" 

# Define your columns
SCORE_COLUMN = "Score" 
ID_COLUMN = "Question" # Used alongside 'Multiplier' as the composite key

def load_agg_data(base_path):
    data_mapping = {}
    if not os.path.exists(base_path):
        print(f"Error: Directory '{base_path}' not found.")
        return data_mapping

    for subdir_name in os.listdir(base_path):
        subdir_path = os.path.join(base_path, subdir_name)
        if os.path.isdir(subdir_path):
            behavior_file = os.path.join(subdir_path, f"{subdir_name}_behavior_agg.csv")
            utility_file = os.path.join(subdir_path, f"{subdir_name}_utility_agg.csv")
            
            subdir_data = {}
            if os.path.exists(behavior_file):
                subdir_data['behavior'] = pd.read_csv(behavior_file)
            if os.path.exists(utility_file):
                subdir_data['utility'] = pd.read_csv(utility_file)
            
            if subdir_data:
                data_mapping[subdir_name] = subdir_data

    return data_mapping

if __name__ == "__main__":
    gemma_data = load_agg_data(base_dir)
    
    # --- PASS 1: Collect global bad composite keys: (Question, Multiplier) ---
    global_bad_behavior_keys = set()
    global_bad_utility_keys = set()
    
    for subdir_name, subdir_data in gemma_data.items():
        if 'behavior' in subdir_data:
            b_df = subdir_data['behavior']
            if SCORE_COLUMN in b_df.columns and ID_COLUMN in b_df.columns and 'Multiplier' in b_df.columns:
                # Find rows where score is 0
                bad_mask = b_df[SCORE_COLUMN] == 0
                # Zip the question and multiplier together into tuples and add to our set
                bad_tuples = set(zip(b_df.loc[bad_mask, ID_COLUMN], b_df.loc[bad_mask, 'Multiplier']))
                global_bad_behavior_keys.update(bad_tuples)
                
        if 'utility' in subdir_data:
            u_df = subdir_data['utility']
            if SCORE_COLUMN in u_df.columns and ID_COLUMN in u_df.columns and 'Multiplier' in u_df.columns:
                # Find rows where score is 0
                bad_mask = u_df[SCORE_COLUMN] == 0
                # Zip the question and multiplier together into tuples and add to our set
                bad_tuples = set(zip(u_df.loc[bad_mask, ID_COLUMN], u_df.loc[bad_mask, 'Multiplier']))
                global_bad_utility_keys.update(bad_tuples)

    print(f"Found {len(global_bad_behavior_keys)} unique (Question, Multiplier) pairs to drop globally in Behavior.")
    print(f"Found {len(global_bad_utility_keys)} unique (Question, Multiplier) pairs to drop globally in Utility.")

    # --- PASS 2: Filter by composite key and calculate means ---
    behavior_avg_scores = {}
    utility_avg_scores = {}
    
    for subdir_name, subdir_data in gemma_data.items():
        
        # Calculate averages for Behavior
        if 'behavior' in subdir_data:
            b_df = subdir_data['behavior']
            if SCORE_COLUMN in b_df.columns and ID_COLUMN in b_df.columns and 'Multiplier' in b_df.columns:
                # Create a multi-index from the current dataframe to check against our bad keys
                current_keys = pd.MultiIndex.from_arrays([b_df[ID_COLUMN], b_df['Multiplier']])
                mask = current_keys.isin(global_bad_behavior_keys)
                
                # Keep only rows that are NOT in the bad keys mask
                clean_b_df = b_df[~mask]
                b_avg = clean_b_df.groupby('Multiplier')[SCORE_COLUMN].mean().to_dict()
                behavior_avg_scores[subdir_name] = b_avg

        # Calculate averages for Utility
        if 'utility' in subdir_data:
            u_df = subdir_data['utility']
            if SCORE_COLUMN in u_df.columns and ID_COLUMN in u_df.columns and 'Multiplier' in u_df.columns:
                # Create a multi-index from the current dataframe to check against our bad keys
                current_keys = pd.MultiIndex.from_arrays([u_df[ID_COLUMN], u_df['Multiplier']])
                mask = current_keys.isin(global_bad_utility_keys)
                
                # Keep only rows that are NOT in the bad keys mask
                clean_u_df = u_df[~mask]
                u_avg = clean_u_df.groupby('Multiplier')[SCORE_COLUMN].mean().to_dict()
                utility_avg_scores[subdir_name] = u_avg

    # --- Print the Results ---
    print("\n=== BEHAVIOR AVERAGE SCORES ===")
    for subdir, scores in behavior_avg_scores.items():
        print(f"{subdir}: {scores}")
        
    print("\n=== UTILITY AVERAGE SCORES ===")
    for subdir, scores in utility_avg_scores.items():
        print(f"{subdir}: {scores}")

# Define your list of baseline directory names here
baseline_subdirs = ["mistral-7b-6", "mistral-7b-all"] # Update with your actual folder names

# Dictionary to hold nested results: {baseline: {mode: {comparison_subdir: results}}}
results_mcnemar = {}

for baseline_name in baseline_subdirs:
    results_mcnemar[baseline_name] = {'behavior': {}, 'utility': {}}
    
    for mode in ['behavior', 'utility']:
        # Use the appropriate global bad keys for the current mode
        bad_keys = global_bad_behavior_keys if mode == 'behavior' else global_bad_utility_keys
        
        # Ensure the baseline actually exists and has data for this mode
        if baseline_name not in gemma_data or mode not in gemma_data[baseline_name]:
            print(f"Warning: Baseline '{baseline_name}' missing '{mode}' data. Skipping.")
            continue
            
        # 1. Prepare the Baseline Data
        base_df = gemma_data[baseline_name][mode]
        base_keys = pd.MultiIndex.from_arrays([base_df[ID_COLUMN], base_df['Multiplier']])
        clean_base = base_df[~base_keys.isin(bad_keys)]
        
        # 2. Iterate through all other subdirectories to compare against the current baseline
        for subdir_name, subdir_data in gemma_data.items():
            if subdir_name == baseline_name:
                continue # Skip comparing the baseline to itself
                
            if mode in subdir_data:
                comp_df = subdir_data[mode]
                
                # Clean the comparison data
                comp_keys = pd.MultiIndex.from_arrays([comp_df[ID_COLUMN], comp_df['Multiplier']])
                clean_comp = comp_df[~comp_keys.isin(bad_keys)]
                
                # 3. Merge Baseline and Comparison on the composite key
                merged = pd.merge(
                    clean_base[[ID_COLUMN, 'Multiplier', SCORE_COLUMN]],
                    clean_comp[[ID_COLUMN, 'Multiplier', SCORE_COLUMN]],
                    on=[ID_COLUMN, 'Multiplier'],
                    suffixes=('_base', '_comp')
                )
                
                if not merged.empty:
                    # 4. Create Contingency Table
                    contingency_table = pd.crosstab(
                        merged[f'{SCORE_COLUMN}_base'], 
                        merged[f'{SCORE_COLUMN}_comp']
                    )
                    
                    # Force the table to be square (McNemar-Bowker requirement)
                    all_cats = sorted(set(contingency_table.index) | set(contingency_table.columns))
                    contingency_table = contingency_table.reindex(index=all_cats, columns=all_cats, fill_value=0)
                    
                    # 5. Perform the Test (requires at least a 2x2 table)
                    if contingency_table.shape[0] > 1: 
                        st = SquareTable(contingency_table.values)
                        test_result = st.symmetry()
                        
                        results_mcnemar[baseline_name][mode][subdir_name] = {
                            "statistic": test_result.statistic,
                            "p_value": test_result.pvalue,
                            "n": len(merged)
                        }
                    else:
                        results_mcnemar[baseline_name][mode][subdir_name] = {
                            "error": "Not enough variance (1x1 table)", 
                            "n": len(merged)
                        }

# --- Print Results ---
print("\n=== McNEMAR-BOWKER TEST RESULTS ===")
for baseline_name, modes in results_mcnemar.items():
    print(f"\n==================================================")
    print(f"BASELINE: {baseline_name}")
    print(f"==================================================")
    
    for mode in ['behavior', 'utility']:
        print(f"\n-- {mode.upper()} --")
        if not modes[mode]:
            print("  No comparisons made.")
        
        for subdir, res in modes[mode].items():
            if "error" in res:
                print(f"  vs {subdir}: {res['error']} (n={res['n']})")
            else:
                sig = "SIGNIFICANT shift" if res['p_value'] < 0.05 else "not significant"
                print(f"  vs {subdir}: Stat={res['statistic']:.4f}, p={res['p_value']:.4e} ({sig}, n={res['n']})")

Found 667 unique (Question, Multiplier) pairs to drop globally in Behavior.
Found 286 unique (Question, Multiplier) pairs to drop globally in Utility.

=== BEHAVIOR AVERAGE SCORES ===
mistral-7b-all: {-2.0: 1.2439024390243902, -1.5: 1.1898734177215189, -1.0: 1.2580645161290323, -0.5: 1.6494845360824741, 0.0: 2.3585858585858586, 0.5: 3.7763975155279503, 1.0: 4.672727272727273, 1.5: 3.524137931034483, 2.0: 3.0353982300884956}
mistral-7b-all-exp-similarity-abs: {-2.0: 1.146341463414634, -1.5: 1.2531645569620253, -1.0: 1.5053763440860215, -0.5: 1.7835051546391754, 0.0: 2.3585858585858586, 0.5: 2.9130434782608696, 1.0: 3.9515151515151516, 1.5: 4.268965517241379, 2.0: 4.557522123893805}
mistral-7b-all-exp-similarity-abs-exponent: {-2.0: 1.1219512195121952, -1.5: 1.3037974683544304, -1.0: 1.5053763440860215, -0.5: 2.020618556701031, 0.0: 2.3585858585858586, 0.5: 2.8509316770186337, 1.0: 3.6787878787878787, 1.5: 4.627586206896551, 2.0: 4.663716814159292}
mistral-7b-6: {-2.0: 1.0609756097560976